# 📚 LangChain ile Retrieval Augmented Generation'a Giriş 🦜🔗

Bu not defterinde LangChain kullanarak Retrieval Augmented Generation'ı nasıl kullanacağınızı öğreneceksiniz.

Kendi belgelerimiz hakkında sorular sormak için bir LLM kullanacağız!

## ⚙️ Kurulum

👉 Temel kütüphaneleri içe aktarmak için aşağıdaki hücreyi çalıştırın.

In [1]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

👉 API anahtarımızı tekrar yüklemek için aşağıdaki hücreyi çalıştırın:

In [2]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## 📚 Neden RAG?

Bir LLM kendi başına, öğrendiği her şey hakkında sorulara yanıt verebilir.

Bunun birkaç dezavantajı vardır:
- Eğitim verileri geçmişten gelir ve en son verilerle güncellenmez.
- Sadece eğitim aldığı verileri bilir.

Bir LLM'yi kendi verilerimizle çalışması için kullanmak istiyoruz. İşte bu noktada RAG (Retrieval-Augmented Generation) devreye girer.

1. **Retrieval-Augmented Generation (RAG)**, gerçek doğruluğu artırmak için bir dil modelini belge alıcı ile birleştirir.
2. **İlgili dış belgeleri alır** (örneğin, bilgi tabanından) yanıtlar üretmeden önce.
3. **Dil modeli hem istemi hem de alınan bağlamı kullanarak** daha bilgili ve temelli çıktılar üretir.

## 🇪🇺 Bağlam

Bu meydan okumada, Avrupa Parlamentosu'ndan belgelerle çalışacağız.

Bir gazeteci olduğunuzu ve Avrupa Parlamentosu'nun genel kurul oturumları sırasında belirli bir konu hakkında neler söylendiğini öğrenmek istediğinizi düşünün. Bu oturumlar yılda 12 kez Strasbourg'da gerçekleşir ve 4 gün sürer. Oturumların transkriptleri EP'nin web sitesinde mevcuttur.

Kesinlikle tüm bu transkriptleri karıştırmak istemezsiniz. O halde, hayatımızı kolaylaştırmak için RAG'ı kullanalım!

Bu, her zaman test etmek için yepyeni veriler alabileceğimiz için çalışmak üzere iyi verilerdir.

## 📘 Verileri alalım

1. [EP'nin web sitesine](https://www.europarl.europa.eu/plenary/en/debates-video.html) gidin. 
1. Bu sizi en son genel kurul oturumuna yönlendirecektir.
1. İlk tarihin altında, "▶️ Verbatim reports HTML" bölümünde `HTML`'e tıklayın.
1. Sayfanın sonuna kaydırın ve alttaki PDF dosyasını indirin.
1. Dosyayı `data` klasörüne kaydedin.

Bir belgeyle başlayacağız, ancak daha sonrası için diğer birkaç günün aynısını şimdiden indirebilirsiniz.

Belgeye bir göz atın. Kaç sayfası var? Belge hakkında bir fikir edinmek için hızlıca belgede gezinin.

## 🔢 Belgeleri gömme

Belgeleri gömmek, tüm belgeleri veya belge parçalarını vektörlere çevirmek anlamına gelir.

LangChain🦜🔗 yine çok yardımcı olacak.

Bir gömme aracı (embedder) başlatalım ve deneyelim. LLM olarak Gemini kullandığımız için, Google'ın metin gömme araçlarında kalalım.

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

👉 Basit bir metin parçasını gömmek için gömme aracının `.embed_query()` metodunu deneyin.

In [4]:
# Embed a text like "What is the capital of France?" and save it to a variable `sample_embedding`

sample_text = "What is the capital of France?"
sample_embedding = embeddings.embed_query(sample_text)

👉 Bu `sample_embedding`'i keşfetmek için zaman ayırın. Nasıl görünüyor? Tipi nedir? Gömme boyutu nedir?

In [5]:
print(f"Type of embedding: {type(sample_embedding)}")
print(f"First 5 elements: {sample_embedding[:5]}")
print(f"Embedding dimension: {len(sample_embedding)}")

Type of embedding: <class 'list'>
First 5 elements: [-0.032554302364587784, 0.013054960407316685, 0.015067406930029392, -0.07128717750310898, -0.03106236644089222]
Embedding dimension: 3072


## 💾 PDF'den gerçek verilerimizi yükle

Artık bir gömmenin nasıl göründüğünü biliyoruz, gerçek verilerimizle çalışmanın zamanı geldi.

👉 [LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs) gidin ve PyPDF kullanarak bir PDF'yi nasıl yükleyebileceğinizi öğrenin.

👉 Sonra devam edin ve daha önce indirdiğiniz PDF'lerden birini yükleyin.

In [6]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/home/ataka/S19D3-S-data-rag-with-langchain/data/TA-9-2024-0076_EN.pdf" 

loader = PyPDFLoader(file_path)

pages = loader.load()

👉 `pages`'i keşfedin:
- Veri tipi nedir?
- Kaç sayfanız var?
- Bir sayfanın tipi nedir?
- Bir sayfanın içeriğine nasıl erişebilirsiniz?
- Tam belgenin kaç karakteri var?
- Bir sayfanın `metadata`'sında neler var?

In [7]:

print(f"Type of pages: {type(pages)}")

print(f"Number of pages: {len(pages)}")

print(f"Type of a page: {type(pages[0])}")

print(f"Content preview: {pages[0].page_content[:200]}...")

total_characters = sum([len(page.page_content) for page in pages])
print(f"Total characters: {total_characters}")

print(f"Metadata: {pages[0].metadata}")

Type of pages: <class 'list'>
Number of pages: 12
Type of a page: <class 'langchain_core.documents.base.Document'>
Content preview: European Parliament
2019-2024
TEXTS ADOPTED
P9_TA(2024)0076
Implementation report on the EU LGBTIQ Equality Strategy 2020-2025
European Parliament resolution of 8 February 2024 on the implementation o...
Total characters: 31526
Metadata: {'producer': 'Aspose.Words for Java 19.12', 'creator': 'Microsoft Office Word', 'creationdate': '2024-06-14T09:03:00+00:00', 'title': 'TA', 'author': 'CROSSFIELD Clare', 'moddate': '2024-06-14T09:03:00+00:00', 'source': '/home/ataka/S19D3-S-data-rag-with-langchain/data/TA-9-2024-0076_EN.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}


## ✂️ Verilerimizi böl

Tam belgemiz gömülmek için çok uzun. Metin gömme aracımız 2.048 tokena kadar giriş alabilir. Gemini modelleri için bu yaklaşık 8.196 karakterdir (token başına 4 karakter).

Bu yüzden belgemizi daha küçük parçalara bölmek istiyoruz.

Zaten çalışabileceğimiz bir dizi sayfamız var. Ama sayfa sonları biraz keyfi: genellikle cümlenin ortasında görünürler.

Ayrıca, sayfalar arasında örtüşme yoktur. Bu yüzden bir sayfanın ilk satırı önceki tüm bağlamı kaçırır. Tam metni biraz örtüşmeyle bölmek daha iyidir.

İlk olarak, PDF'yi tekrar yükleyeceğiz, bu sefer bölmeden.

In [8]:
loader = PyPDFLoader(file_path, mode='single')
pdf = loader.load()
pdf_text = pdf[0].page_content
len(pdf_text)

31548

Artık tüm PDF'imizi tek bir belge olarak aldığımıza göre, onu daha akıllı bir şekilde parçalara bölebiliriz.

👉 Yine, ["Özyinelemeli olarak bölme" konusundaki LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) gidin ve `pdf` _belgelerimizi_ parçalara (LangChain'de `documents` olarak adlandırılır) nasıl böleceğinizi öğrenin.

2_000 karakter (bizim durumumuzda yaklaşık yarım sayfa) parçalara 400 örtüşmeyle bölün. İsterseniz diğer değerlerle deneyebilirsiniz.

`RecursiveCharacterTextSplitter`'ın `.split_documents()` metodunu kullanın: bu metod giriş olarak bir belge alır ve bölünmüş belgeler çıktısı verir.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,      
    chunk_overlap=400,    
    add_start_index=True 
)

all_splits = text_splitter.split_documents(pages)

👉 `all_splits`'i inceleyin:
- Veri tipi nedir?
- Kaç bölümünüz var?
- Bir bölümün tipi nedir?
- Bir bölümün içeriğine nasıl erişebilirsiniz?
- Şimdi toplamda kaç karakterimiz var?
- Bir bölümün `metadata`'sında neler var?

In [10]:

print(f"Type of all_splits: {type(all_splits)}")

print(f"Number of splits: {len(all_splits)}")

print(f"Content of first split: {all_splits[0].page_content[:200]}...")

total_split_chars = sum([len(doc.page_content) for doc in all_splits])
print(f"Total characters in splits: {total_split_chars}")

print(f"Metadata of first split: {all_splits[0].metadata}")

Type of all_splits: <class 'list'>
Number of splits: 22
Content of first split: European Parliament
2019-2024
TEXTS ADOPTED
P9_TA(2024)0076
Implementation report on the EU LGBTIQ Equality Strategy 2020-2025
European Parliament resolution of 8 February 2024 on the implementation o...
Total characters in splits: 35140
Metadata of first split: {'producer': 'Aspose.Words for Java 19.12', 'creator': 'Microsoft Office Word', 'creationdate': '2024-06-14T09:03:00+00:00', 'title': 'TA', 'author': 'CROSSFIELD Clare', 'moddate': '2024-06-14T09:03:00+00:00', 'source': '/home/ataka/S19D3-S-data-rag-with-langchain/data/TA-9-2024-0076_EN.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'start_index': 0}


## 🗄️ Her şeyi bir araya getir: belgelerimizi gömme ve vektör deposunda sakla

Elimizde şunlar var:
- Bir gömme aracı
- Veriyi yüklemek için bir yükleyici
- Belgemizi belgelere bölmek için bir metin bölücü

Neyi kaçırıyoruz?

Belgelerimizi gömebiliriz, ama onları bir yerde saklamak istiyoruz. İşte burada vektör deposu devreye girer: şunları saklamamıza olanak sağlar:
- belgeyi (parçayı),
- onun gömmesini,
- meta verilerini.

Sonraki adımda belgeleri verimli bir şekilde alabilecek olacağız.

👉 Bir `InMemoryVectorStore` nasıl oluşturabileceğinizi görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

In [11]:
# Import the necessary libraries

from langchain_community.vectorstores import InMemoryVectorStore
# Create an in-memory vector store using the embedder `embeddings` we created earlier

vector_store = InMemoryVectorStore(embeddings)
# Add the `all_splits` to the vector store and store the result in a variable called `document_ids`

document_ids = vector_store.add_documents(documents=all_splits)

In [12]:
# Have a look at the first 3 document IDs

print(f"First 3 Document IDs: {document_ids[:3]}")

First 3 Document IDs: ['479507d3-a1c5-46f4-a8d0-5f38ec77bdbb', '4def57ca-a289-4648-aa68-6898be0566da', 'fd6eb6e1-6689-4c63-b373-90ad93c84c99']


In [15]:
# Use the vector store's `get_by_ids` method. You have to give it a list of document IDs.

fetched_docs = vector_store.get_by_ids(document_ids[:1])
fetched_docs

[Document(id='479507d3-a1c5-46f4-a8d0-5f38ec77bdbb', metadata={'producer': 'Aspose.Words for Java 19.12', 'creator': 'Microsoft Office Word', 'creationdate': '2024-06-14T09:03:00+00:00', 'title': 'TA', 'author': 'CROSSFIELD Clare', 'moddate': '2024-06-14T09:03:00+00:00', 'source': '/home/ataka/S19D3-S-data-rag-with-langchain/data/TA-9-2024-0076_EN.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='European Parliament\n2019-2024\nTEXTS ADOPTED\nP9_TA(2024)0076\nImplementation report on the EU LGBTIQ Equality Strategy 2020-2025\nEuropean Parliament resolution of 8 February 2024 on the implementation of the EU \nLGBTIQ Equality Strategy 2020-2025 (2023/2082(INI))\nThe European Parliament,\n– having regard to the Charter of Fundamental Rights of the European Union,\n– having regard to Article 2 of the Treaty on European Union (TEU),\n– having regard to the European Convention on Human Rights (ECHR) and the related \ncase law of the European Court of Hum

👉 Bir vektör deposundaki belgenin içeriğine ve meta verilerine nasıl erişebilirsiniz?

In [14]:
if fetched_docs:
    doc = fetched_docs[0]
    print(f"Content: {doc.page_content[:100]}...") 
    print(f"Metadata: {doc.metadata}")

Content: European Parliament
2019-2024
TEXTS ADOPTED
P9_TA(2024)0076
Implementation report on the EU LGBTIQ E...
Metadata: {'producer': 'Aspose.Words for Java 19.12', 'creator': 'Microsoft Office Word', 'creationdate': '2024-06-14T09:03:00+00:00', 'title': 'TA', 'author': 'CROSSFIELD Clare', 'moddate': '2024-06-14T09:03:00+00:00', 'source': '/home/ataka/S19D3-S-data-rag-with-langchain/data/TA-9-2024-0076_EN.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'start_index': 0}


## 🔎 Benzer belgeleri almak için vektör deposunu kullan

Artık belgeleri gömleğe çevirdiğimize göre, benzer belgeleri almak için vektör deposunu kullanabiliriz.

👉 Bunun nasıl çalıştığını görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

Bir sorgu kullanın, örneğin "Tarım politikası üzerine tartışmayı özetle.", ve en benzer belgeleri bulun. Ayrıca alınacak belge sayısını da belirtebilirsiniz.

In [16]:
# Save your question into a variable called `query`

query = "Summarize the debate on agriculture policy."
# Use the vector store to find similar documents to the query. Store the result in a variable called `retrieved_docs`

retrieved_docs = vector_store.similarity_search(query, k=3)
print(f"Retrieved {len(retrieved_docs)} documents.")
print(f"First doc preview: {retrieved_docs[0].page_content[:150]}...")

Retrieved 3 documents.
First doc preview: 73. Instructs its President to forward this resolution to the Council and the Commission, the 
governments and parliaments of the Member States and ca...


Bu, RAG'ın sözde "Alma" (Retrieval) kısmını tamamlar: artık sorgumuza en benzer belgeleri bulabiliriz.

Çalışmanın çoğu artık tamamlandı!

## 💬 Sorumuza bir cevap üret

Şimdiye kadar benzer belgeleri almamızı sağlamak için sadece bir **gömme modeli** kullandık.

Şimdi, sorumuzla bir cevap almak için üretici bir LLM kullanacağız: ona aldığımız belgeler ve sorumuzla besleyeceğiz.

Bunu yapmanın en temel yolu tüm girdilerimizi birbirine bağlamak, sorumuzla eklemek ve sonucu görmek olacaktır.

Bir deneyelim.

👉 İlk olarak önceki meydan okumalarda olduğu gibi bir LLM başlatın.

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0)

Sonra temel bir istem oluşturun:

In [23]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Şimdi istemi kullanın:

In [24]:
response = llm.invoke(prompt)
print(response.content)

Based on the text provided, there is no debate or information regarding agriculture policy.

The document focuses entirely on:
*   **LGBTIQ+ rights** and their mainstreaming across all EU policies.
*   **Anti-discrimination** measures, including on grounds of sexual orientation, gender identity, and intersectionality.
*   **Humanitarian aid** sensitivity.
*   The recognition of **parenthood** (including for same-sex parents) across EU Member States.


Bu fena değil, ama modele daha fazla rehberlik vererek daha kapsamlı bir istem yazarak daha iyisini yapabiliriz.

Bunu yapan ilk kişiler biz değilmişiz ve LangChain'in bizim için önceden hazırlanmış istem kütüphanesi var.

👉 Aşağıdaki hücreyi çalıştırın ve nasıl çalıştığını anlamaya çalışın. (LangSmithMissingAPIKeyWarning hakkında bir uyarı alacaksınız, bunu görmezden gelebilirsiniz.)

In [25]:
from langchain_classic import hub

prompt_template = hub.pull("rlm/rag-prompt")

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("\n")
print(example_messages[0].content)



You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


LangChain'in bizim için nasıl daha kesin bir istem oluşturduğunu görüyor musunuz? Bunu RAG'ımız için kullanalım!

👉 İlk olarak, tüm alınan belgeleri iki yeni satırla ayrılmış tek bir uzun dizgiye birleştirin.

In [26]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

formatted_context = format_docs(retrieved_docs)

👉 Sonra, sorgunuz ve alınan belgelerden başlayarak bir `prompt` oluşturun. Yukarıdaki örneğe bakmayı unutmayın.

In [27]:
final_prompt = prompt_template.invoke(
    {"context": formatted_context, "question": query}
)
print(final_prompt)

messages=[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: Summarize the debate on agriculture policy. \nContext: 73. Instructs its President to forward this resolution to the Council and the Commission, the \ngovernments and parliaments of the Member States and candidate countries, and the \nsubnational parliaments and local authorities of the Member States and candidate \ncountries.\n\n24. Recalls that European humanitarian aid should be gender-, age-, protection- and \nLGBTIQ+-sensitive, while taking into consideration intersectionality as a cross-cutting \nprinciple, and in line with humanitarian principles;\n25. Underlines that the EU must leave no one behind in the protection of fundamental \nrights;\nRecommendations\n26. Calls for the EU and the Member States t

👉 Son olarak az önce oluşturduğumuz `the_prompt` ile LLM modelini kullanın:

In [28]:
final_response = llm.invoke(final_prompt)

print("--- RAG Sonucu (Profesyonel Prompt) ---")
print(final_response.content)

--- RAG Sonucu (Profesyonel Prompt) ---
I don't know. The provided context does not contain information about agriculture policy. The text focuses on LGBTIQ+ rights, anti-discrimination, and humanitarian aid within the European Union.


🎉 İlk RAG'ımızı tamamladık: LLM kendisine sağladığımız belgelerde ***temelli*** metin üretti.

## 💾 Gömmelerimizi kalıcı hale getir

Şimdiye kadar bellekte vektör deposuyla çalıştık. Bu yüzden not defterinizi kapattığınızda, tüm gömmeleri de kaybedeceksiniz.

⚠️ Bu gömmelerin sağlayıcınızın platformunda, bu durumda Google'ın makinelerinde çalışan modeller tarafından üretildiğini unutmayın. Ve bedava çalışmazlar. 💰

Bunun gibi bir, nispeten küçük belge için maliyet düşüktür, ama hızla artar. Şimdiye kadar sadece bir günün transkriptleriyle çalıştık. Oturum başına 3 tane daha, yılda 12 oturum, birden fazla yıl var...

Bunu çözmek için sadece vektör depomuzla kalıcı bir taneyi değiştireceğiz. Bu LangChain'in avantajıdır: bileşenleri değiştirmek çok kolay.

Bellekteki vektör depomuz deneme için harikaydı, şimdi başka bir taneyle değiştireceğiz. Çok popüler bir vektör deposu olan [Chroma](https://www.trychroma.com/)'yı kullanacağız. Bunu yerel olarak çalıştırabilir ve LangChain aracılığıyla kullanabiliriz.

Tüm akışımızı yeniden oluşturacağız. Her şeyi birkaç kod hücresinde tekrar bir araya getirmeye çalışmak iyi bir alıştırmadır. Aynı zamanda her şeyi yeniden kullanılabilir koda dönüştüreceğiz.

Sonunda iki fonksiyon istiyoruz:

1. `embed_and_store()`: Başka bir oturumun transkriptini vektör veritabanımıza ekle, böylece alacağımız daha fazla veri olsun.
2. `answer()`: Vektör depomuzla farklı sorularla sorgula.

#### 1. Bir Chroma vektör deposu başlat

👉 **Veri kalıcılığıyla** (yani verileri diskteki bir dizinde saklayarak) Chroma vektör deposunun nasıl oluşturulacağını görmek için [LangChain'in belgelerine](https://python.langchain.com/docs/integrations/vectorstores/chroma/) bakın.

In [29]:
from langchain_chroma import Chroma
persist_directory = "chroma_db"
vector_store = Chroma(
    collection_name="europarl_collection",
    embedding_function=embeddings,
    persist_directory=persist_directory
)

print(f"Chroma veritabanı '{persist_directory}' klasörüne bağlandı.")

Chroma veritabanı 'chroma_db' klasörüne bağlandı.


#### 2. `embed_and_store()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [32]:
# 20. Görevin Çözümü: embed_and_store fonksiyonu

def embed_and_store(file_path, vector_store):
    """
    Bir PDF dosyasını yükler, parçalara böler ve vektör deposuna kaydeder.
    
    Args:
        file_path (str): PDF dosyasının yolu.
        vector_store (Chroma): Kayıt yapılacak vektör veritabanı objesi.
        
    Returns:
        list: Eklenen belgelerin ID listesi.
    """
    
    # 1. YÜKLEME (Extract)
    # Loader her çağrıldığında yeni dosya için sıfırdan oluşturulmalı
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    
    # 2. BÖLME (Transform)
    # Global olarak tanımladığımız text_splitter'ı kullanabiliriz veya burada yeniden tanımlayabiliriz.
    # Güvenli olması için burada aynı ayarlarla kullanalım.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=400,
        add_start_index=True
    )
    all_splits = text_splitter.split_documents(pages)
    
    # 3. YÜKLEME / GÖMME (Load)
    # add_documents metodu, metinleri embed eder ve veritabanına yazar.
    document_ids = vector_store.add_documents(all_splits)
    
    print(f"Başarılı! {len(document_ids)} parça belge '{file_path}' dosyasından eklendi.")
    
    return document_ids

👉 Fonksiyonunuzu bir dosya veya hatta iki dosyayla deneyin:

In [33]:
test_file_path = "/home/ataka/S19D3-S-data-rag-with-langchain/data/TA-9-2024-0076_EN.pdf" # Dosya adınız neyse
ids = embed_and_store(test_file_path, vector_store)

Başarılı! 22 parça belge '/home/ataka/S19D3-S-data-rag-with-langchain/data/TA-9-2024-0076_EN.pdf' dosyasından eklendi.


#### 3. `answer()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [34]:
# 22. Görevin Çözümü: answer fonksiyonu

def answer(query, vector_store, llm, prompt_template=None):
    """
    Vektör deposunu ve LLM'i kullanarak bir soruya cevap verir.
    
    Args:
        query (str): Kullanıcının sorusu.
        vector_store (Chroma): Arama yapılacak veritabanı.
        llm (ChatModel): Cevabı üretecek yapay zeka modeli.
        prompt_template (PromptTemplate, optional): Özel şablon. Yoksa varsayılan kullanılır.
        
    Returns:
        str: Modelin cevabı.
    """
    
    # 1. GERİ ÇAĞIRMA (Retrieval)
    # Soruyu vektöre çevir ve en benzer 4 belgeyi bul.
    retrieved_docs = vector_store.similarity_search(query, k=4)
    
    # 2. BAĞLAM OLUŞTURMA
    # Bulunan belgeleri metin formatına çevir.
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    
    # 3. PROMPT HAZIRLAMA
    # Eğer şablon verilmediyse hub'dan çek (veya basitçe oluştur)
    if not prompt_template:
        # Kodun hızlı çalışması için hub.pull yerine manuel tanımlayalım, 
        # hub bazen API key hatası verebilir.
        from langchain_core.prompts import ChatPromptTemplate
        prompt_template = ChatPromptTemplate.from_template(
            "You are an assistant for question-answering tasks. "
            "Use the following pieces of retrieved context to answer the question. "
            "If you don't know the answer, just say that you don't know. "
            "Use three sentences maximum and keep the answer concise.\n\n"
            "Question: {question} \n\n"
            "Context: {context} \n\n"
            "Answer:"
        )

    # Şablonu doldur
    final_prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # 4. ÜRETİM (Generation)
    response = llm.invoke(final_prompt)
    
    return response.content

👉 Fonksiyonunuzu beğendiğiniz bir sorguyla deneyin:

In [35]:
# 23. Görevin Çözümü: answer fonksiyonunu test etme

my_query = "What is the main topic of the debate?"
answer_result = answer(my_query, vector_store, llm)

print(f"Soru: {my_query}")
print("-" * 30)
print(f"Cevap: {answer_result}")

Soru: What is the main topic of the debate?
------------------------------
Cevap: Based on the context, the main topic of the debate is the rights and equality of LGBTIQ+ people within the European Union. The discussion focuses on the implementation of the LGBTIQ Equality Strategy 2020-2025 and the ongoing discrimination and violence faced by the community. It also addresses specific issues, such as the call for a ban on conversion practices.


🏁 Tebrikler! Artık LangChain kullanarak RAG'da ustalaştınız ve vektör deponuza daha fazla belge eklemek ve onu sorgulamak için yeniden kullanılabilir fonksiyonlar yapmayı öğrendiniz.

## [İsteğe Bağlı] Meta veri ekleme

Kurduğumuz RAG, vektör deposundaki tüm belgeleri sorgular. Orada birden fazla yılın bilgisinin olduğunu düşünün. Yıllara veya tarihlere göre filtreyebilsek kullanışlı olurdu, değil mi?

Bunu nasıl yaparız? Vektör deposundaki belgelerin meta veri içerdiğini unutmayın. Eğer tarihi ekleyebilseydik, daha sonra filtrelemek için kullanabilirdik.

İpucu: Meta verilerinizi pipeline'ınızda olabildiğince erken ekleyin. Verileriniz vektör deposunda saklandıktan sonra eklemeye çalışmayın.

👉 `embed_and_store()` fonksiyonunuzu uyarlayın.

In [36]:
# 24. Görevin Çözümü: Metadata ekleyerek kayıt eden fonksiyon

def embed_and_store_fancy(file_path, vector_store, session_date):
    """
    PDF yükler, böler ve her parçaya 'session_date' metadatasını ekleyerek kaydeder.
    """
    # 1. Yükle
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    
    # 2. Böl
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
    all_splits = text_splitter.split_documents(pages)
    
    # 3. METADATA EKLE (Kritik Adım)
    # Her bir parçanın metadata sözlüğüne tarihi ekliyoruz.
    for split in all_splits:
        # Mevcut metadata'yı koru, üzerine tarihi ekle
        split.metadata['session_date'] = session_date
    
    # 4. Kaydet
    document_ids = vector_store.add_documents(all_splits)
    
    print(f"'{session_date}' tarihli {len(document_ids)} belge eklendi.")
    return document_ids

👉 Fonksiyonunuzu deneyin ve vektör deponuzun ek meta veri içerdiğini kontrol edin.

In [37]:
embed_and_store_fancy("/home/ataka/S19D3-S-data-rag-with-langchain/data/TA-9-2024-0076_EN.pdf", vector_store, session_date="2023-10-02")

'2023-10-02' tarihli 22 belge eklendi.


['d794a11d-97ac-4a22-970d-a5276b08322d',
 '33297d91-ef2d-4b05-994e-3c506fc76003',
 'd60e0ac5-536e-4c9e-9d4c-a0624d739a32',
 '3eec91a6-fe85-452a-86c4-fa1346deb92c',
 '0b5aafe6-c524-48a0-ad09-bb5454c0c0f6',
 '61d65f33-1860-48e7-ac90-e548224fa18a',
 'feaedf4f-0780-4934-a318-3ad3fd8b1cc3',
 '73d88e1c-96c9-453d-9098-6aeb47fbb559',
 '922efb91-661b-4341-873f-d6d84f864664',
 'a1a61cae-e82b-449b-8eb3-292f09a12106',
 '826e818f-3028-4360-bc87-5664b8b1404b',
 '642e15cb-db81-489f-a39e-6f68fe608496',
 '0c5d70d1-63bd-4695-9665-ae9653de4345',
 'd8c921b2-aae0-4958-824c-f11479b83a8c',
 'b30891dc-e638-47a6-8761-56e5cf2f38a7',
 '981e8915-8d9e-4b45-9178-9c7ea7c7dcee',
 '268a0a09-f776-4019-ac81-b6fa8ed17b55',
 'ffdfa2ec-9418-4375-aded-fe8a42d190b8',
 'b2f0c76f-9b76-4dad-af00-df1965e137b8',
 '9480f8bd-dcd7-4b3f-b2d2-eb5af8a77624',
 'a60f4bc8-5338-488a-9f03-8a3d89d98d7f',
 'e009e0f4-fa9c-4c4c-8916-e2aadf0be258']

Şimdi alıcıyı kullanıcının sorduğu tarihe göre sınırlamamız gerekiyor.

👉 `answer()` fonksiyonunuzu bir tarih alabilecek ve yeni meta verilere dayalı olarak belgeleri filtreleyebilecek şekilde uyarlayın.

In [38]:
# 26. Görevin Çözümü: Filtreli cevap fonksiyonu

def answer_with_filter(query, vector_store, llm, target_date=None):
    # Filtre sözlüğü oluştur
    # Eğer tarih verildiyse Chroma'nın anlayacağı filtre formatını hazırla
    search_kwargs = {}
    if target_date:
        # Metadata'daki 'session_date' alanı 'target_date'e eşit olanları getir.
        search_kwargs['filter'] = {'session_date': target_date}
        search_kwargs['k'] = 4 # Kaç belge geleceği
    else:
        search_kwargs['k'] = 4

    # similarity_search metoduna filtreyi gönderiyoruz (**kwargs açılımı ile veya direkt parametre ile)
    # LangChain Chroma implementasyonunda genellikle .similarity_search(query, k=..., filter=...) kullanılır.
    
    if target_date:
         retrieved_docs = vector_store.similarity_search(
             query, 
             k=4,
             filter={'session_date': target_date}
         )
    else:
         retrieved_docs = vector_store.similarity_search(query, k=4)

    # --- Buradan sonrası standart RAG akışı ---
    if not retrieved_docs:
        return "Belirtilen kriterlere uygun doküman bulunamadı."

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    
    from langchain_core.prompts import ChatPromptTemplate
    prompt_template = ChatPromptTemplate.from_template(
        "Context: {context} \n\n Question: {question} \n\n Answer:"
    )
    
    final_prompt = prompt_template.invoke({"context": docs_content, "question": query})
    response = llm.invoke(final_prompt)
    
    return response.content

In [39]:
# 27. Görevin Çözümü: Filtreli test

# 1. Tarih vererek sor
print("--- Tarih Filtreli Sonuç ---")
print(answer_with_filter("What happened regarding agriculture?", vector_store, llm, target_date="2023-10-02"))

# 2. Olmayan bir tarih vererek sor (Boş dönmeli veya bulamadım demeli)
print("\n--- Olmayan Tarih Sonucu ---")
print(answer_with_filter("What happened?", vector_store, llm, target_date="1990-01-01"))

--- Tarih Filtreli Sonuç ---
Based on the provided text, there is no mention of agriculture. The document is a European Parliament resolution focused on the implementation of the EU LGBTIQ Equality Strategy 2020-2025.

--- Olmayan Tarih Sonucu ---
Belirtilen kriterlere uygun doküman bulunamadı.


Harika! Güçlü bir RAG sistemi oluşturmak için benzerlik aramasını meta veri aramasıyla birleştirdiniz!